## 2 — Gráficos: Visão Geral do Campus Restinga

**Seção 2.4 — Indicadores e matrículas:**

| Gráfico | Tipo | Descrição |
|---------|------|-----------|
| 1 | Linha | Evolução de TC, TE, TR, TPE e IEf por ano |
| 2 | Linha | Evolução de MREG e MRET por ano |
| 3 | Barras empilhadas | Matrículas atendidas por ano e categoria |
| 4 | Barras empilhadas | Matrículas por curso e ano |
| 5 | Linha | Ingressantes e Concluintes por ano |
| 6 | Linha (facetas) | Ingressantes e Concluintes por curso e ano |

**Seção 2.5 — Evasão:**

| Gráfico | Tipo | Descrição |
|---------|------|-----------|
| 7 | Barras empilhadas | Motivos de saída por ano |
| 8 | Barras horizontais empilhadas % | Proporção dos motivos de evasão por curso (último ano) |

**Seção 2.6 — Conclusão:**

| Gráfico | Tipo | Descrição |
|---------|------|-----------|
| 9  | Barras empilhadas | Conclusões no prazo vs. com atraso por curso |
| 10 | Barras horizontais | Tempo mediano de conclusão por curso |
| 11 | Boxplot | Distribuição do tempo até conclusão por curso |


### 2.1. Importações e configurações

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option("display.max_columns", None)

# Paleta de cores por categoria de situação
CORES_CATEGORIA = {
    "Em curso":    "#4CAF50",
    "Concluintes": "#2196F3",
    "Evadidos":    "#F44336",
}

# Paleta de cores por situação de matrícula detalhada
CORES_SITUACAO = {
    "Ingressante":          "#A5D6A7",
    "Em fluxo":             "#4CAF50",
    "Retido":               "#FF9800",
    "Concluída no prazo":   "#2196F3",
    "Concluída com atraso": "#64B5F6",
    "Abandono":             "#F44336",
    "Desligada":            "#E91E63",
    "Transf. externa":      "#9E9E9E",
    "Transf. interna":      "#BDBDBD",
    "Integralizada":        "#00BCD4",
}

# Paleta de cores por indicador percentual
CORES_INDICADORES = {
    "TC":       "#4E79A7",  # azul
    "TE":       "#E15759",  # vermelho
    "TR":       "#F28E2B",  # laranja
    "IEf":      "#59A14F",  # verde
    "TPE":      "#B07AA1",  # lilás
}

CORES_CURSO = {
    "Análise e Desenvolvimento de Sistemas": "#5E60CE",  # azul violeta
    "Eletrônica Industrial":                "#E63946",  # vermelho
    "Gestão Desportiva e de Lazer":         "#2A9D8F",  # verde água
    "Letras - Português e Espanhol":        "#9B5DE5",  # roxo
    "Técnico em Comércio":                  "#F4A261",  # laranja
    "Técnico em Eletrônica":                "#00B4D8",  # ciano
    "Técnico em Guia de Turismo":           "#EF476F",  # rosa
    "Técnico em Informática":               "#8AC926",  # verde limão
    "Técnico em Lazer":                     "#C77DFF",  # lilás
    "Processos Gerenciais":                 "#FFBE0B",  # amarelo
    "Técnico em Agroecologia":              "#4361EE",  # azul
}
import plotly.graph_objects as go

# ── Paletas locais para gráficos de evasão e conclusão (light mode) ──────────
# Cada cor pertence a uma família de matiz distinta: sem tons similares.
CORES_SAIDA = {
    "Abandono":        "#D62828",  # vermelho
    "Desligada":       "#F77F00",  # laranja
    "Transf. externa": "#023E8A",  # azul escuro marinho
    "Transf. interna": "#2D6A4F",  # verde escuro
    "Reprovada":       "#7B2D8B",  # roxo
}

CORES_CONCLUSAO = {
    "Concluída no prazo":   "#2D6A4F",  # verde escuro
    "Concluída com atraso": "#E76F51",  # laranja queimado
}

# Verde = dentro do prazo previsto | Laranja = além do prazo
COR_DENTRO_PRAZO = "#2D6A4F"
COR_ALEM_PRAZO   = "#E76F51"


### 2.2 Carga dos dados tratados

In [2]:
df_dados_tratados = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")

print("Shape:", df_dados_tratados.shape)
df_dados_tratados.head(3)

Shape: (10445, 19)


,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno,Concluinte_Previsto
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino,False
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino,False
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino,False


In [3]:
df_dados_tratados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10445 entries, 0 to 10444
Data columns (total 19 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ano                              10445 non-null  int64         
 1   Categoria da Situação            10445 non-null  object        
 2   Cor / Raça                       10445 non-null  object        
 3   Código da Matricula              10445 non-null  int64         
 4   Código do Ciclo Matricula        10445 non-null  int64         
 5   Data de Fim Previsto do Ciclo    10445 non-null  datetime64[ns]
 6   Data de Inicio do Ciclo          10445 non-null  datetime64[ns]
 7   Data de Ocorrencia da Matricula  10445 non-null  datetime64[ns]
 8   Eixo Tecnológico                 10445 non-null  object        
 9   Faixa Etária                     10445 non-null  object        
 10  Mês De Ocorrência da Situação    10445 non-null  datetime6

### 2.3. Cálculo dos indicadores

In [4]:
def calcular_indicadores(df, agrupamento):

    df_indicadores = (
        df.groupby(agrupamento)
        .agg(
            ingressantes    = ("Situação de Matrícula", lambda x: (x == "Ingressante").sum()),
            em_curso        = ("Categoria da Situação",  lambda x: (x == "Em curso").sum()),
            concluintes     = ("Categoria da Situação",  lambda x: (x == "Concluintes").sum()),
            evadidos        = ("Categoria da Situação",  lambda x: (x == "Evadidos").sum()),
            matr_atendidas  = ("Código da Matricula",    "count"),
            conc_no_prazo   = ("Situação de Matrícula",  lambda x: (x == "Concluída no prazo").sum()),
            conc_com_atraso = ("Situação de Matrícula",  lambda x: (x == "Concluída com atraso").sum()),
            abandono        = ("Situação de Matrícula",  lambda x: (x == "Abandono").sum()),
            desligados      = ("Situação de Matrícula",  lambda x: (x == "Desligada").sum()),
            transf_ext      = ("Situação de Matrícula",  lambda x: (x == "Transf. externa").sum()),
            transf_int      = ("Situação de Matrícula",  lambda x: (x == "Transf. interna").sum()),
            integralizadas  = ("Situação de Matrícula",  lambda x: (x == "Integralizada").sum()),
            conc_previstos  = ("Concluinte_Previsto",    "sum"),
            MREG            = ("Situação de Matrícula",
                               lambda x: ((x == "Em fluxo") | (x == "Ingressante")).sum()),
            MRET            = ("Situação de Matrícula",  lambda x: (x == "Retido").sum()),
        )
        .reset_index()
    )

    # Evita divisão por zero substituindo 0 por NaN antes de dividir
    ma = df_indicadores["matr_atendidas"].replace(0, np.nan)

    # TC — Taxa de Conclusão
    df_indicadores["TC"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]) / ma * 100
    )

    # TE — Taxa de Evasão
    df_indicadores["TE"] = (
        (df_indicadores["abandono"] + df_indicadores["desligados"] + df_indicadores["transf_ext"])
        / ma * 100
    )

    # TR — Taxa de Retenção
    df_indicadores["TR"] = df_indicadores["MRET"] / ma * 100

    # IEf — Índice de Eficiência
    # Numerador: todos os concluintes (no prazo + com atraso)
    # Denominador: matrículas que tiveram algum desfecho (todas as saídas)
    matr_finalizadas = (
        df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]
        + df_indicadores["abandono"] + df_indicadores["desligados"]
        + df_indicadores["transf_int"] + df_indicadores["transf_ext"]
    ).replace(0, np.nan)
    df_indicadores["IEf"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"])
        / matr_finalizadas * 100
    )
    
    # TMREG — Taxa de Matrícula Continuada Regular
    df_indicadores["TMREG"] = df_indicadores["MREG"] / ma * 100

    # TPE — Taxa de Permanência e Êxito
    df_indicadores["TPE"] = df_indicadores["TC"] + df_indicadores["TMREG"]


    return df_indicadores.fillna(0).round(2)


# Calcula por Ano e por Ano + Curso
ind_ano       = calcular_indicadores(df_dados_tratados, ["Ano"])
ind_ano_curso = calcular_indicadores(df_dados_tratados, ["Ano", "Nome de Curso"])

print(ind_ano[["Ano", "TC", "TE", "TR", "TPE", "IEf"]])


    Ano    TC     TE     TR    TPE    IEf
0  2017  8.38  15.78   5.18  78.55  34.34
1  2018  7.80   9.84   7.70  82.36  43.96
2  2019  6.07  11.05   5.68  83.19  35.45
3  2020  0.16   7.52  13.57  78.66   2.06
4  2021  2.50   5.70  23.89  70.41  30.43
5  2022  7.48   9.29  34.60  56.11  44.62
6  2023  9.77   5.27  31.94  62.79  64.94
7  2024  5.91   2.32  35.15  62.42  71.81


### 2.4. Gráficos

In [5]:

# Gráfico 1: Evolução de todos os indicadores percentuais por ano
# melt() transforma o DataFrame de formato largo (uma coluna por indicador) para formato longo (uma linha por indicador+ano)

mapeamento_indicadores = {
    "TC": "Taxa de Conclusão",
    "TE": "Taxa de Evasão",
    "TR": "Taxa de Retenção",
    "TPE": "Taxa de Permanência e Êxito",
    "IEf": "Índice de Eficiência",
}

ind_longo = ind_ano.melt(
    id_vars="Ano",
    value_vars=["TC", "TE", "TR", "TPE", "IEf"],
    var_name="Indicador",
    value_name="Valor (%)",
)

ind_longo.head()


,Ano,Indicador,Valor (%)
0,2017,TC,8.38
1,2018,TC,7.80
2,2019,TC,6.07
3,2020,TC,0.16
4,2021,TC,2.50


In [6]:
fig_g1 = px.line(
    ind_longo,
    x="Ano",
    y="Valor (%)",
    color="Indicador",
    markers=True,
    color_discrete_map=CORES_INDICADORES,
    title="Evolução dos Indicadores Acadêmicos de Permanência e Êxito do IFRS do Campus Restinga",
    labels={"Indicador": "", "Valor (%)": "(%)"},
)

# troca apenas o nome exibido na legenda
for trace in fig_g1.data:
    trace.name = mapeamento_indicadores.get(trace.name, trace.name)
    trace.hovertemplate = trace.hovertemplate.replace(
        trace.legendgroup,
        mapeamento_indicadores.get(trace.legendgroup, trace.legendgroup)
    )


fig_g1.update_xaxes(tickmode="linear", dtick=1)
fig_g1.update_layout(hovermode="x unified", legend=dict(orientation="h", y=-0.2))

fig_g1.show()

In [7]:
# 2: Evolução de MREG e MRET por ano (linhas)
matr_ativas_longo = ind_ano.melt(
    id_vars="Ano",
    value_vars=["MREG", "MRET"],
    var_name="Tipo",
    value_name="Matrículas",
)

matr_ativas_longo.head()

,Ano,Tipo,Matrículas
0,2017,MREG,569
1,2018,MREG,765
2,2019,MREG,991
3,2020,MREG,960
4,2021,MREG,762


In [8]:
fig_g2 = px.line(
    matr_ativas_longo,
    x="Ano",
    y="Matrículas",
    color="Tipo",
    markers=True,
    color_discrete_map={"MREG": "#4CAF50", "MRET": "#FF9800"},
    labels={"Tipo": ""},
)
fig_g2.update_xaxes(tickmode="linear", dtick=1)
fig_g2.update_layout(hovermode="x unified", legend=dict(orientation="h", y=-0.2))
fig_g2.show()

In [9]:
# Gráfico 3: Matrículas atendidas por ano e categoria

mat_cat = (
    df_dados_tratados.groupby(["Ano", "Categoria da Situação"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

mat_cat.head()

,Ano,Categoria da Situação,Qtd
0,2017,Concluintes,70
1,2017,Em curso,611
2,2017,Evadidos,130
3,2018,Concluintes,80
4,2018,Em curso,844


In [10]:
fig_g3 = px.bar(
    mat_cat,
    x="Ano",
    y="Qtd",
    color="Categoria da Situação",
    barmode="stack",
    color_discrete_map=CORES_CATEGORIA,
    title="Matrículas Atendidas por Ano e Categoria de Matrícula",
    labels={"Qtd": "Matrículas", "Categoria da Situação": ""},
    text_auto=True,
)
fig_g3.update_xaxes(tickmode="linear", dtick=1)

fig_g3.update_layout(
    legend=dict(
        orientation="h",   # horizontal
        yanchor="top",
        y=-0.2,            # posição abaixo do gráfico
        xanchor="center",
        x=0.5
    )
)

fig_g3.show()

In [11]:
# Gráfico 4: Matrículas atendidas por curso e ano

mat_curso_ano = (
    df_dados_tratados.groupby(["Ano", "Nome de Curso"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

mat_curso_ano.head()

,Ano,Nome de Curso,Qtd
0,2017,Análise e Desenvolvimento de Sistemas,247
1,2017,Eletrônica Industrial,84
2,2017,Gestão Desportiva e de Lazer,112
3,2017,Letras - Português e Espanhol,30
4,2017,Técnico em Comércio,33


In [12]:
fig_g4 = px.bar(
    mat_curso_ano,
    x="Ano",
    y="Qtd",
    color="Nome de Curso",
    color_discrete_map=CORES_CURSO,
    barmode="stack",
    title="Matrículas Atendidas por Ano e Curso",
    labels={"Qtd": "Matrículas", "Nome de Curso": "Curso"},
)
fig_g4.update_xaxes(tickmode="linear", dtick=1)

fig_g4.show()

In [13]:
# Gráfico 5: Ingressantes e Concluintes por ano

fluxo_longo = ind_ano.melt(
    id_vars="Ano",
    value_vars=["ingressantes", "concluintes"],
    var_name="Tipo",
    value_name="Quantidade",
)

# Renomeia os rótulos para ficarem mais legíveis na legenda
fluxo_longo["Tipo"] = fluxo_longo["Tipo"].map({
    "ingressantes": "Ingressantes",
    "concluintes":  "Concluintes",
})

fluxo_longo.head()

,Ano,Tipo,Quantidade
0,2017,Ingressantes,292
1,2018,Ingressantes,344
2,2019,Ingressantes,373
3,2020,Ingressantes,218
4,2021,Ingressantes,147


In [14]:
fig_g5 = px.line(
    fluxo_longo,
    x="Ano",
    y="Quantidade",
    color="Tipo",
    markers=True,
    color_discrete_map={
        "Ingressantes": "#7E57C2",
        "Concluintes":  "#2196F3",
    },
    title="Ingressantes e Concluintes por Ano",
    labels={"Tipo": "",  "Quantidade": "Matrículas"},
)
fig_g5.update_xaxes(tickmode="linear", dtick=1)
fig_g5.update_layout(hovermode="x unified", legend=dict(orientation="h", y=-0.2))

fig_g5.show()

In [15]:
# Gráfico 6: Ingressantes e Concluintes por curso e por ano
# Permite comparar o funil de entrada vs. saída em cada curso separadamente

# Prepara os dados: soma ingressantes por Ano e Curso a partir dos dados brutos
ingressantes_curso = (
    df_dados_tratados[df_dados_tratados["Situação de Matrícula"] == "Ingressante"]
    .groupby(["Ano", "Nome de Curso"])["Código da Matricula"]
    .count()
    .reset_index(name="Quantidade")
)
ingressantes_curso["Tipo"] = "Ingressantes"

# Soma concluintes por Ano e Curso a partir dos dados brutos
concluintes_curso = (
    df_dados_tratados[df_dados_tratados["Categoria da Situação"] == "Concluintes"]
    .groupby(["Ano", "Nome de Curso"])["Código da Matricula"]
    .count()
    .reset_index(name="Quantidade")
)
concluintes_curso["Tipo"] = "Concluintes"

# Junta os dois DataFrames em um só para o plotly express
fluxo_curso = pd.concat([ingressantes_curso, concluintes_curso], ignore_index=True)

fluxo_curso.head()

,Ano,Nome de Curso,Quantidade,Tipo
0,2017,Análise e Desenvolvimento de Sistemas,59,Ingressantes
1,2017,Eletrônica Industrial,27,Ingressantes
2,2017,Gestão Desportiva e de Lazer,29,Ingressantes
3,2017,Letras - Português e Espanhol,25,Ingressantes
4,2017,Técnico em Comércio,33,Ingressantes


In [16]:
# facet_col="Nome de Curso" cria um painel (faceta) separado para cada curso,
# permitindo comparar o funil de cada um sem sobreposição de linhas
fig_g6 = px.line(
    fluxo_curso,
    x="Ano",
    y="Quantidade",
    color="Tipo",
    facet_col="Nome de Curso",
    facet_col_wrap=3,
    markers=True,
    color_discrete_map={
        "Ingressantes": "#7E57C2",
        "Concluintes":  "#2196F3",
    },
    title="Ingressantes e Concluintes por Curso e Ano",
    labels={"Tipo": "", "Quantidade": "Quantidade"},
)
fig_g6.update_xaxes(tickmode="linear", dtick=2)
fig_g6.update_layout(
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.12),
    height=700,
)
# Remove o prefixo "Nome de Curso=" dos títulos de cada painel
fig_g6.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig_g6.show()

### 2.5. Evasão


#### Gráfico 7 — Motivos de Saída por Ano


In [ ]:
# Gráfico 7: Motivos de saída por ano (barras empilhadas, valores absolutos)
#
# Usamos valores absolutos (não %) porque a MAGNITUDE importa aqui:
# queremos saber se o número de abandonos cresceu, não apenas a proporção.
# Cada cor representa uma família de matiz distinta.

situacoes_saida = ["Abandono", "Desligada", "Transf. externa", "Transf. interna"]

df_saidas = (
    df_dados_tratados[df_dados_tratados["Situação de Matrícula"].isin(situacoes_saida)]
    .groupby(["Ano", "Situação de Matrícula"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

fig_g7 = px.bar(
    df_saidas,
    x="Ano", y="Qtd",
    color="Situação de Matrícula",
    barmode="stack",
    color_discrete_map=CORES_SAIDA,
    title="7 — Motivos de Saída por Ano",
    labels={"Qtd": "Matrículas", "Situação de Matrícula": "Motivo"},
    text_auto=True,
)
fig_g7.update_xaxes(tickmode="linear", dtick=1)
fig_g7.update_layout(legend=dict(orientation="h", y=-0.2))
fig_g7.show()


#### Gráfico 8 — Proporção dos Motivos de Evasão por Curso (último ano)


In [ ]:
# Gráfico 8: Proporção (%) de cada motivo de evasão por curso — último ano
#
# Usamos PROPORÇÃO (%) aqui porque os cursos têm tamanhos muito diferentes;
# comparar volumes absolutos seria enganoso.
# O último ano evita que cursos extintos ou fases históricas distintas
# distorçam a comparação do perfil atual.
# Barras horizontais: nomes de cursos ficam mais legíveis no eixo Y.
# Cursos ordenados pelo total de evadidos (maior problema aparece no topo).

ANO_REFERENCIA = int(df_dados_tratados["Ano"].max())
motivos_ev = ["Abandono", "Desligada", "Transf. externa"]

ev_2024 = df_dados_tratados[
    (df_dados_tratados["Ano"] == ANO_REFERENCIA)
    & (df_dados_tratados["Situação de Matrícula"].isin(motivos_ev))
]

ev_curso = (
    ev_2024
    .groupby(["Nome de Curso", "Situação de Matrícula"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)
tot_ev = ev_curso.groupby("Nome de Curso")["Qtd"].sum().reset_index(name="Total")
ev_curso = ev_curso.merge(tot_ev, on="Nome de Curso")
ev_curso["Pct"] = (ev_curso["Qtd"] / ev_curso["Total"] * 100).round(1)

# Ordena pelo total de evadidos: curso com mais evasões aparece no topo
ordem = tot_ev.sort_values("Total", ascending=True)["Nome de Curso"].tolist()

fig_g8 = px.bar(
    ev_curso,
    y="Nome de Curso", x="Pct",
    color="Situação de Matrícula",
    orientation="h",
    barmode="stack",
    color_discrete_map=CORES_SAIDA,
    text_auto=".1f",
    title=f"8 — Proporção dos Motivos de Evasão por Curso ({ANO_REFERENCIA})",
    labels={"Pct": "(%)", "Nome de Curso": "", "Situação de Matrícula": "Motivo"},
    category_orders={"Nome de Curso": ordem},
    custom_data=["Total"],
)
fig_g8.update_traces(
    hovertemplate="<b>%{y}</b><br>%{fullData.name}: %{x:.1f}%<br>Total evadidos: %{customdata[0]}<extra></extra>"
)
fig_g8.update_layout(
    xaxis=dict(range=[0, 100], title="(%)"),
    legend=dict(orientation="h", y=-0.2),
)
fig_g8.show()

print(f"\nTotal de evadidos por curso em {ANO_REFERENCIA}:")
print(tot_ev.sort_values("Total", ascending=False).to_string(index=False))


### 2.6. Conclusão


#### Preparo: DataFrame de Concluídos e Cálculo do Tempo de Conclusão

**Como é calculado o tempo de conclusão?**

Para cada matrícula com situação *Concluída no prazo* ou *Concluída com atraso*,
calcula-se:

```
Anos_Conclusao = (Mês De Ocorrência da Situação − Data de Inicio do Ciclo) / 365,25
```

- **Data de Inicio do Ciclo**: data em que o ciclo de matrícula começou (ingresso efetivo no curso).
- **Mês De Ocorrência da Situação**: mês/ano em que a conclusão foi registrada na PNP.
- O resultado é o número de anos (com decimais) que o estudante levou para concluir.

**Filtros aplicados:**
- Apenas matrículas com duração positiva e inferior a 15 anos (remove inconsistências).
- **Não** filtramos por Ano_Ingresso >= 2017, pois estudantes que ingressaram antes
  de 2017 e concluíram após também estão registrados na PNP e devem ser contabilizados.

**Mediana vs. Média:** usamos a **mediana** porque ela é robusta a outliers
(um estudante que levou 10 anos não distorce o resultado).


In [ ]:
# Prepara df_conc: calcula o tempo de conclusão para cada matrícula concluída.
# Inclui TODOS os anos disponíveis (não filtra por Ano_Ingresso >= 2017)
# para não excluir estudantes que ingressaram antes de 2017 mas concluíram depois.

df_conc = df_dados_tratados[
    df_dados_tratados["Situação de Matrícula"].isin(
        ["Concluída no prazo", "Concluída com atraso"]
    )
].copy()

df_conc["Data de Inicio do Ciclo"] = pd.to_datetime(
    df_conc["Data de Inicio do Ciclo"], errors="coerce"
)
df_conc["Mês De Ocorrência da Situação"] = pd.to_datetime(
    df_conc["Mês De Ocorrência da Situação"], errors="coerce"
)

# Anos entre início do ciclo e data de conclusão
df_conc["Anos_Conclusao"] = (
    (df_conc["Mês De Ocorrência da Situação"] - df_conc["Data de Inicio do Ciclo"])
    .dt.days / 365.25
).round(2)

# Remove durações inválidas (negativas ou biologicamente impossíveis > 15 anos)
df_conc = df_conc[
    (df_conc["Anos_Conclusao"] > 0)
    & (df_conc["Anos_Conclusao"] < 15)
].copy()

print(f"Concluídos com tempo calculado: {len(df_conc)}")
print("\nConcluídos por curso:")
print(
    df_conc.groupby(["Nome de Curso", "Situação de Matrícula"])["Anos_Conclusao"]
    .agg(N="count", Mediana="median", Media="mean")
    .round(2)
    .sort_values(["Nome de Curso", "Situação de Matrícula"])
    .to_string()
)


#### Gráfico 9 — Conclusões no Prazo vs. Com Atraso por Curso


In [ ]:
# Gráfico 9: Conclusões no prazo vs. com atraso por curso
# Base: df_conc (todos os concluídos registrados na PNP, sem filtro de período)

conc_curso = (
    df_conc
    .groupby(["Nome de Curso", "Situação de Matrícula"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

# Ordena pelo total de concluídos (maior volume no topo)
total_conc = conc_curso.groupby("Nome de Curso")["Qtd"].sum().reset_index(name="Total")
ordem_conc = total_conc.sort_values("Total", ascending=False)["Nome de Curso"].tolist()

fig_g9 = px.bar(
    conc_curso,
    x="Nome de Curso", y="Qtd",
    color="Situação de Matrícula",
    barmode="stack",
    color_discrete_map=CORES_CONCLUSAO,
    title="9 — Conclusões: No Prazo vs. Com Atraso por Curso (2017–2024)",
    labels={"Qtd": "Concluintes", "Nome de Curso": "", "Situação de Matrícula": "Situação"},
    text_auto=True,
    category_orders={"Nome de Curso": ordem_conc},
)
fig_g9.update_layout(
    xaxis_tickangle=-30,
    legend=dict(orientation="h", y=-0.25),
)
fig_g9.show()


#### Gráfico 10 — Tempo Mediano de Conclusão por Curso


In [ ]:
# Gráfico 10: Tempo mediano de conclusão por curso (barras horizontais)
#
# Barra VERDE  = mediana dentro do prazo previsto para o tipo de curso
# Barra LARANJA = mediana além do prazo previsto → retenção sistemática
#
# Referências: 2 anos (técnico) | 4 anos (superior/tecnologia)
# Ordenado do maior atraso (topo) para o menor: os problemas ficam visíveis primeiro.

REF_TECNICO  = 2.0
REF_SUPERIOR = 4.0

tempo_mediano = (
    df_conc
    .groupby(["Nome de Curso", "Tipo de Curso"])["Anos_Conclusao"]
    .agg(Mediana="median", Media="mean", N="count")
    .reset_index()
    .round(2)
    .sort_values("Mediana", ascending=True)   # ascending=True → maior no topo do barh
)

def cor_barra(row):
    ref = REF_TECNICO if "Técnico" in str(row["Tipo de Curso"]) else REF_SUPERIOR
    return COR_DENTRO_PRAZO if row["Mediana"] <= ref else COR_ALEM_PRAZO

tempo_mediano["Cor"]     = tempo_mediano.apply(cor_barra, axis=1)
tempo_mediano["Legenda"] = tempo_mediano["Cor"].map({
    COR_DENTRO_PRAZO: "Dentro do prazo previsto",
    COR_ALEM_PRAZO:   "Além do prazo previsto",
})

fig_g10 = px.bar(
    tempo_mediano,
    x="Mediana", y="Nome de Curso",
    orientation="h",
    color="Legenda",
    color_discrete_map={
        "Dentro do prazo previsto": COR_DENTRO_PRAZO,
        "Além do prazo previsto":   COR_ALEM_PRAZO,
    },
    text="Mediana",
    title="10 — Tempo Mediano de Conclusão por Curso (anos após o ingresso)",
    labels={"Mediana": "Mediana (anos)", "Nome de Curso": "", "Legenda": ""},
    custom_data=["Media", "N", "Tipo de Curso"],
)
fig_g10.update_traces(
    texttemplate="%{x:.2f} anos",
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Mediana: %{x:.2f} anos<br>"
        "Média: %{customdata[0]:.2f} anos<br>"
        "N concluídos: %{customdata[1]}<br>"
        "Tipo: %{customdata[2]}<extra></extra>"
    ),
)
fig_g10.add_vline(
    x=REF_TECNICO, line_dash="dash", line_color="gray", line_width=1.5,
    annotation_text="2 anos (técnico)", annotation_position="top",
    annotation_font_size=10,
)
fig_g10.add_vline(
    x=REF_SUPERIOR, line_dash="dot", line_color="#023E8A", line_width=1.5,
    annotation_text="4 anos (superior)", annotation_position="top",
    annotation_font_size=10,
)
fig_g10.update_layout(
    xaxis=dict(range=[0, tempo_mediano["Mediana"].max() * 1.25]),
    legend=dict(orientation="h", y=-0.15),
)
fig_g10.show()

print("\nTabela resumo (Mediana, Média, N concluídos):")
print(tempo_mediano[["Nome de Curso", "Tipo de Curso", "Mediana", "Media", "N"]]
      .sort_values("Mediana", ascending=False).to_string(index=False))


#### Gráfico 11 — Distribuição do Tempo até Conclusão por Curso (Boxplot)


In [ ]:
# Gráfico 11: Boxplot do tempo até conclusão por curso
#
# O boxplot complementa o gráfico 10:
#   - Mediana (linha central): tempo típico de conclusão
#   - Caixa (IQR): onde estão os 50% centrais dos estudantes
#   - Bigodes: variação geral
#   - Pontos isolados: outliers (casos extremos)
# Um boxplot LARGO indica heterogeneidade: alguns concluem rápido, outros muito tarde.
# Cursos superiores: linha de referência em 4 anos | Técnicos: 2 anos

# Ordena do maior atraso mediano para o menor (problemas ficam no topo)
ordem_box = (
    df_conc.groupby("Nome de Curso")["Anos_Conclusao"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig_g11 = px.box(
    df_conc,
    x="Nome de Curso", y="Anos_Conclusao",
    color="Nome de Curso",
    color_discrete_map=CORES_CURSO,
    points="outliers",
    title="11 — Distribuição do Tempo até Conclusão por Curso (anos após o ingresso)",
    labels={"Anos_Conclusao": "Anos após o Ingresso", "Nome de Curso": ""},
    category_orders={"Nome de Curso": ordem_box},
)
fig_g11.update_layout(showlegend=False, xaxis_tickangle=-30)
fig_g11.add_hline(
    y=2, line_dash="dash", line_color="gray", line_width=1.5,
    annotation_text="2 anos (técnico)", annotation_position="right",
    annotation_font_size=10,
)
fig_g11.add_hline(
    y=4, line_dash="dot", line_color="#023E8A", line_width=1.5,
    annotation_text="4 anos (superior)", annotation_position="right",
    annotation_font_size=10,
)
fig_g11.show()
